# EDS Form Data Extraction
## Using Claude API with Batch Processing and Prompt Caching

This notebook processes Executive Document Summary (EDS) forms and extracts structured data.

**Features:**
- Batch API processing for 50% cost savings
- Prompt caching for 90% savings on repeated prompts
- Hybrid extraction approach (main query + targeted checkbox queries)
- Test mode (single-page PDFs) and Production mode (multi-page PDFs with metadata)
- Professional services contracts filter (optional)
- Clobber toggle to skip/overwrite existing results
- Individual JSON output per form
- Automatic batch chunking for large datasets

## 1. Setup and Configuration

In [85]:
import os
import json
import base64
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Set
import pandas as pd
from pdf2image import convert_from_path
from PIL import Image
import io
import anthropic
from datetime import datetime

jloc = '../../../'

# Load API key from file
with open(jloc + '.claude_key', 'r') as f:
    api_key = f.read().strip()

# Configuration
CONFIG = {
    # Model selection
    'model':'claude-haiku-4-5-20251001',
    # 'claude-haiku-4-5-20251001',  # Haiku for cost-effectiveness
    # Other options:
    # 'claude-sonnet-4-5'  # Sonnet 4.5 for better accuracy (3x cost)
    # 'claude-opus-4-20250514'      # Opus 4 for highest accuracy (15x cost)
    
    'max_tokens': 2000,
    'clobber': False,  # Set to True to overwrite existing results
    
    # Directories
    'test_dir': jloc + 'data/raw/_exampleforms',  # Single-page EDS forms for testing
    'production_dir': jloc + 'data/raw/_contracts',  # Multi-page contract PDFs
    'metadata_csv': jloc + 'code/preprocessing/zero_shot_results_full_corpus.csv',  # CSV with filename, page_number columns
    'prof_contract_csv': jloc + 'data/raw/indiana_prof_services_contracts.csv',  # CSV showing which files are for prof services contracts
    'output_dir': jloc + 'data/intermediate_products/eds_claude_json_production',  # Individual JSON results
    'api_key': api_key,
    
    # Processing mode
    'mode': 'production',  # 'test' or 'production'
    'prof_services_only': True,  # Set to True to only process professional services contracts
    'limit_files': 0,  # Limit number of files to process (None or 0 for no limit)
    
    # Batch monitoring
    'batch_check_interval_seconds': 300,  # How often to check batch status (default: 300 = 5 minutes)
}

# Create output directory
Path(CONFIG['output_dir']).mkdir(exist_ok=True)

print(f"Configuration loaded:")
print(f"  Mode: {CONFIG['mode']}")
print(f"  Clobber: {CONFIG['clobber']}")
print(f"  Model: {CONFIG['model']}")
print(f"  Prof Services Only: {CONFIG['prof_services_only']}")
print(f"  Limit Files: {CONFIG['limit_files'] if CONFIG['limit_files'] else 'No limit'}")
print(f"  Batch Check Interval: {CONFIG['batch_check_interval_seconds']}s")
print(f"  Output: {CONFIG['output_dir']}")

Configuration loaded:
  Mode: production
  Clobber: False
  Model: claude-haiku-4-5-20251001
  Prof Services Only: True
  Limit Files: No limit
  Batch Check Interval: 300s
  Output: ../../../data/intermediate_products/eds_claude_json_production


## 2. Helper Functions

In [87]:
def pdf_page_to_base64(pdf_path: str, page_num: int = 1) -> Tuple[str, int, int]:
    """
    Convert a specific page of a PDF to base64-encoded PNG.
    Returns: (base64_string, width, height)
    """
    # Convert PDF page to image - reduced DPI to 150 for smaller file sizes
    images = convert_from_path(
        pdf_path,
        first_page=page_num,
        last_page=page_num,
        dpi=150  # Reduced from 200 - still readable for forms
    )
    
    if not images:
        raise ValueError(f"Could not extract page {page_num} from {pdf_path}")
    
    img = images[0]
    
    # Resize if needed (keep under 1568px on long edge)
    max_dimension = 1568
    width, height = img.size
    
    if max(width, height) > max_dimension:
        scale = max_dimension / max(width, height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
        width, height = new_width, new_height
    
    # Convert to base64 using JPEG with compression for smaller size
    # Forms are typically black text on white, so JPEG works well
    buffer = io.BytesIO()
    # Convert RGBA to RGB if needed for JPEG
    if img.mode in ('RGBA', 'LA', 'P'):
        # Create white background
        background = Image.new('RGB', img.size, (255, 255, 255))
        if img.mode == 'P':
            img = img.convert('RGBA')
        background.paste(img, mask=img.split()[-1] if img.mode in ('RGBA', 'LA') else None)
        img = background
    
    # Save as JPEG with quality=85 for good balance of size/quality
    img.save(buffer, format='JPEG', quality=85, optimize=True)
    img_bytes = buffer.getvalue()
    base64_str = base64.b64encode(img_bytes).decode('utf-8')
    
    return base64_str, width, height


def estimate_image_tokens(width: int, height: int) -> int:
    """Estimate tokens for an image based on dimensions."""
    return int((width * height) / 750)


def get_output_path(input_path: str, output_dir: str) -> str:
    """Generate output JSON path from input path."""
    filename = Path(input_path).stem
    return str(Path(output_dir) / f"{filename}.json")


def should_process(output_path: str, clobber: bool) -> bool:
    """Check if file should be processed based on clobber setting."""
    if clobber:
        return True
    return not Path(output_path).exists()


def save_result(output_path: str, data: Dict):
    """Save extraction result to JSON file."""
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=2)


print("Helper functions loaded.")

Helper functions loaded.


## 3. Prompt Templates with Caching

We'll use prompt caching to reuse the system instructions across all forms, saving 90% on those tokens.

In [89]:
# System prompt for main extraction (will be cached)
SYSTEM_PROMPT_MAIN = """
You are an expert at extracting structured data from government forms. You will be analyzing Executive Document Summary (EDS) forms.

Your task is to extract ALL text-based fields from the form. Focus on clearly printed or typed text fields.

Extract the following fields when present:
- EDS Number
- Date prepared
- Agency name and address
- Requisition number
- Agency contact information (name, telephone, email)
- Vendor/Courier information (name, ID, address, email, telephone)
- Account numbers and names
- Contract amounts (this action, new contract total, revenue generated)
- Fiscal year information and amounts
- Time period covered (from/to dates)
- Statutory authority
- Description of work
- Justification of vendor selection
- Explanation if submitted late
- Any approval dates and signatures

IMPORTANT VALIDATION RULES:
- Use empty arrays [] when no items are found, not null
- Use null only for single-value fields when not found
- Ensure all array fields are valid JSON arrays

Return ONLY valid JSON in this exact format (no markdown, no explanations):
{
    "eds_number": "string or null",
    "date_prepared": "string or null",
    "agency_name": "string or null",
    "agency_address": "string or null",
    "requisition_number": "string or null",
    "agency_contact_name": "string or null",
    "agency_contact_phone": "string or null",
    "agency_contact_email": "string or null",
    "vendor_name": "string or null",
    "vendor_id": "string or null",
    "vendor_address": "string or null",
    "vendor_email": "string or null",
    "vendor_phone": "string or null",
    "account_numbers": ["string"],
    "account_names": ["string"],
    "contract_amount_this_action": "string or null",
    "contract_amount_new_total": "string or null",
    "revenue_generated": "string or null",
    "fiscal_year": "string or null",
    "fiscal_year_amount": "string or null",
    "time_period_from": "string or null",
    "time_period_to": "string or null",
    "statutory_authority": "string or null",
    "description_of_work": "string or null",
    "justification_vendor_selection": "string or null",
    "explanation_if_late": "string or null",
    "approval_dates": ["string"],
    "signature_dates": ["string"],
    "approver_names": ["string"]
}
"""

# System prompt for contract type extraction (will be cached)
SYSTEM_PROMPT_CONTRACT_TYPE = """
You are an expert at identifying selections on government forms that use UNDERSCORES with X marks.

IMPORTANT: This form does NOT use traditional checkbox squares (☐). Instead, options are shown
with underscores, and selections are indicated by an "X" mark positioned ABOVE or to the LEFT of
the underscore or option label.

Example of SELECTED option:    Example of UNSELECTED option:
     X
  ______                          ______

Your task is to identify which CONTRACT TYPE and METHOD OF SOURCE SELECTION options
are marked (have X marks) on Executive Document Summary (EDS) forms.

FORM LAYOUT:
The form has a "CONTRACTS & LEASES" section with options arranged in TWO COLUMNS:

LEFT COLUMN:
- Professional/Personal Services
- Grant
- Lease
- Attorney
- MOU
- QPA

RIGHT COLUMN:
- Contract for procured Services
- Maintenance
- License Agreement
- Amendment#
- Renewal #
- Other

There can only be one of Professional/Personal Services, Grant, Lease, Attorney, MOU, Contract for
procured Services, Maintenance, or License Agreement selected. QPA, Renewal, Amendment, and Other
can co-occur with other selections.

Below that is a "METHOD OF SOURCE SELECTION" section with these options:
- Bid/Quotation
- RFP#
- Emergency
- Other (specify)
- Negotiated
- Special Procurement
There can only be one "METHOD OF SOURCE SELECTION" option selected.

CRITICAL RULES FOR DETECTING X MARKS AND SELECTIONS:
1. An option is MARKED if there is an "X" mark positioned ABOVE the underscore or to the LEFT of the option label
2. An option with a number or text filled in after it (like "QPA 10120" or "Renewal # ___2___") indicates
   that option IS marked - the filled-in value is evidence of selection
3. Blank underscores with no X mark above them are NOT marked
4. Do NOT confuse option labels with X marks - "MOU" is a label, not a selection mark

SPATIAL AWARENESS FOR X MARKS:
- An X mark belongs to the option whose underscore is DIRECTLY BELOW the X
- If an X appears between two options vertically, it belongs to the option below it, not above
- X marks may appear to the left of option text as well

COMMON ERRORS TO AVOID:
- Do NOT mark "MOU" just because it contains letters - look for an actual X mark
- Do NOT mark options that are simply listed but have no X mark
- DO mark "QPA" if you see "QPA" followed by a number (the number is the QPA identifier)
- DO mark "Renewal #" or "Amendment#" if followed by a number in the underscore

Be very careful to distinguish between marked and unmarked options.
Give a confidence score (probability value 0.0-1.0) for each returned option.

IMPORTANT VALIDATION RULES:
- Arrays must have matching lengths (contract_types and contract_confidence must be same length)
- Confidence values must be between 0.0 and 1.0
- Use empty arrays [] when nothing is selected, not null

Return ONLY valid JSON in this exact format (no markdown, no explanations):
{
    "contract_types": ["type1", "type2"],
    "contract_confidence": [0.0, 0.0],
    "selection_types": ["type1", "type2"],
    "selection_confidence": [0.0, 0.0]
}
"""

# System prompt for yes/no fields and vendor status extraction (will be cached)
SYSTEM_PROMPT_YES_NO = """
You are an expert at detecting checkmarks, X marks, and selections on government forms.

Your task is to identify YES/NO selections on Executive Document Summary (EDS) forms.

Identify the YES/NO selections for these fields:
- "Is the vendor registered with the Secretary of State?"
- "Is there Renewal Language in the document?"
- "Is there a 'Termination for Convenience' clause in the document?"
- "Will the attached document involve data processing or telecommunications systems?"
- "Is the Primary Vendor Women Owned?"
- "Is the Primary Vendor Minority Owned?"
- "Is the Primary Vendor IN-Veteran Owned?"
- "Is the Sub Vendor Women Owned?"
- "Is the Sub Vendor Minority Owned?"
- "Is the sub Vendor IN-Veteran Owned?"

CRITICAL SPATIAL RULES FOR YES/NO SELECTIONS, X MARKS AND UNDERSCORES:
- An X mark is associated with an option ONLY if the X appears ABOVE the underscore for that option
- The X mark MUST NOT be positioned BELOW the underscore to count for that option
- If an X mark appears between two options, it belongs to the option whose underscore is DIRECTLY BELOW the X, not the option above it
- There can only be one true option, YES or NO, for each field.

Be very careful to distinguish between marked and unmarked options.

IMPORTANT VALIDATION RULES:
- Return "unclear" only when truly ambiguous, not as default

Return ONLY valid JSON in this exact format (no markdown, no explanations):
{
  "vendor_registered": "yes|no|unclear",
  "renewal_language": "yes|no|unclear",
  "termination_clause": "yes|no|unclear",
  "data_processing_involved": "yes|no|unclear",
  "primary_vendor_status": {
    "minority": "yes|no|unclear",
    "women": "yes|no|unclear",
    "in_veteran": "yes|no|unclear|no IN-Veteran option"
  },
  "subvendor_status": {
    "minority": "yes|no|unclear",
    "women": "yes|no|unclear",
    "in_veteran": "yes|no|unclear|no_option"
  }
}
"""

# Minimal user prompts (now that details are in system prompts)
USER_PROMPT_MAIN = "Extract all text fields from this EDS form and return as JSON."

USER_PROMPT_CONTRACT_TYPE = "Identify which contract type(s) and selection method(s) is/are selected on this form."

USER_PROMPT_YES_NO_FIELDS = "Identify the yes/no selections on this form."

print("Prompt templates loaded.")

Prompt templates loaded.


## 4. Batch API Processing Functions

The Batch API allows us to submit multiple requests and get 50% cost savings with 24-hour processing.

In [91]:
# ========================================
# BATCH REGISTRY MANAGEMENT
# ========================================

def get_registry_path() -> Path:
    """Get path to the central batch registry file."""
    return Path(CONFIG['output_dir']) / 'batch_registry.json'


def load_batch_registry() -> Dict:
    """
    Load the central batch registry.
    
    Returns:
        Dict with structure:
        {
            'batches': {
                'batch_id': {
                    'batch_file': str,
                    'timestamp': str,
                    'status': str,
                    'created_at': str,
                    'num_requests': int,
                    'num_forms': int,
                    'retrieved': bool,  # NEW: marks if results have been retrieved
                    'pdfs': [{'pdf_path': str, 'page_num': int, 'form_id': str}]
                }
            },
            'pdf_to_batch': {
                'pdf_path:page_num': 'batch_id'
            }
        }
    """
    registry_path = get_registry_path()
    
    if not registry_path.exists():
        return {'batches': {}, 'pdf_to_batch': {}}
    
    try:
        with open(registry_path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"Warning: Could not load batch registry: {e}")
        return {'batches': {}, 'pdf_to_batch': {}}


def save_batch_registry(registry: Dict):
    """Save the batch registry to disk."""
    registry_path = get_registry_path()
    with open(registry_path, 'w') as f:
        json.dump(registry, f, indent=2)


def register_batch(batch_id: str, batch_file: str, timestamp: str, 
                  status: str, created_at: str, forms_metadata: Dict,
                  num_requests: int):
    """
    Register a new batch in the central registry.
    
    Args:
        batch_id: Batch ID from API
        batch_file: Name of the batch JSONL file
        timestamp: Timestamp string
        status: Batch status
        created_at: ISO format creation time
        forms_metadata: Dict mapping form_id -> (output_path, page_num, pdf_path)
        num_requests: Total number of requests in batch
    """
    registry = load_batch_registry()
    
    # Build PDF list from forms_metadata
    pdfs = []
    for form_id, (output_path, page_num, pdf_path) in forms_metadata.items():
        pdfs.append({
            'pdf_path': pdf_path,
            'page_num': page_num,
            'form_id': form_id
        })
    
    # Register batch
    registry['batches'][batch_id] = {
        'batch_file': batch_file,
        'timestamp': timestamp,
        'status': status,
        'created_at': created_at,
        'num_requests': num_requests,
        'num_forms': len(pdfs),
        'retrieved': False,  # NEW: track if results have been retrieved
        'pdfs': pdfs
    }
    
    # Map PDFs to batch
    for pdf_info in pdfs:
        key = f"{pdf_info['pdf_path']}:{pdf_info['page_num']}"
        registry['pdf_to_batch'][key] = batch_id
    
    save_batch_registry(registry)
    print(f"Registered batch {batch_id} with {len(pdfs)} PDFs")


def mark_batch_retrieved(batch_id: str):
    """
    Mark a batch as retrieved in the registry.
    This prevents it from being retrieved again, even if some forms had errors.
    """
    registry = load_batch_registry()
    
    if batch_id in registry['batches']:
        registry['batches'][batch_id]['retrieved'] = True
        save_batch_registry(registry)
        print(f"Marked batch {batch_id} as retrieved")


def update_batch_status(batch_id: str, status: str):
    """Update the status of a batch in the registry."""
    registry = load_batch_registry()
    
    if batch_id in registry['batches']:
        registry['batches'][batch_id]['status'] = status
        save_batch_registry(registry)


def is_pdf_in_pending_batch(pdf_path: str, page_num: int) -> bool:
    """
    Check if a PDF is already in a pending (unretrieved) batch.
    
    A batch is considered pending if:
    1. It has NOT been marked as retrieved
    
    Returns:
        True if PDF is in a pending batch, False otherwise
    """
    registry = load_batch_registry()
    key = f"{pdf_path}:{page_num}"
    
    if key not in registry['pdf_to_batch']:
        return False
    
    batch_id = registry['pdf_to_batch'][key]
    batch_info = registry['batches'].get(batch_id)
    
    if not batch_info:
        return False
    
    # Check if batch has been retrieved
    # If 'retrieved' key doesn't exist (old registry format), assume not retrieved
    retrieved = batch_info.get('retrieved', False)
    
    return not retrieved


def get_pending_batches() -> List[str]:
    """Get list of batch IDs that have unretrieved results."""
    registry = load_batch_registry()
    
    pending = []
    
    for batch_id, batch_info in registry['batches'].items():
        # Check if batch has been retrieved
        retrieved = batch_info.get('retrieved', False)
        if not retrieved:
            pending.append(batch_id)
    
    return pending


# ========================================
# BATCH REQUEST CREATION
# ========================================

def create_batch_request(custom_id: str, image_base64: str, system_prompt: str, user_prompt: str) -> Dict:
    """
    Create a single batch request object.
    Uses prompt caching by including cache_control in system message.
    """
    return {
        "custom_id": custom_id,
        "params": {
            "model": CONFIG['model'],
            "max_tokens": CONFIG['max_tokens'],
            "system": [
                {
                    "type": "text",
                    "text": system_prompt,
                    "cache_control": {"type": "ephemeral"}  # Enable caching
                }
            ],
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": image_base64
                            }
                        },
                        {
                            "type": "text",
                            "text": user_prompt
                        }
                    ]
                }
            ]
        }
    }


def prepare_batch_requests(forms_to_process: List[Tuple[str, str, int]]) -> Tuple[List[Dict], Dict]:
    """
    Prepare all batch requests for forms.
    Each form gets 3 requests: main extraction + 2 checkbox queries.
    
    Args:
        forms_to_process: List of (pdf_path, output_path, page_num) tuples
    
    Returns:
        Tuple of (batch_requests, forms_metadata)
        - batch_requests: List of batch request objects
        - forms_metadata: Dict mapping form_id -> (output_path, page_num, pdf_path)
    """
    batch_requests = []
    forms_metadata = {}
    skipped_count = 0
    
    for pdf_path, output_path, page_num in forms_to_process:
        print(f"Preparing: {Path(pdf_path).name} (page {page_num})")
        
        try:
            # Convert PDF page to image
            image_base64, width, height = pdf_page_to_base64(pdf_path, page_num)
            tokens = estimate_image_tokens(width, height)
            print(f"  Image: {width}x{height} (~{tokens} tokens)")
        except Exception as e:
            # Skip corrupted or unreadable PDFs
            print(f"  ERROR: Could not process PDF - {str(e)[:100]}")
            print(f"  Skipping this file...")
            skipped_count += 1
            continue
        
        # Generate unique IDs for this form
        form_id = Path(pdf_path).stem
        if page_num > 1:
            form_id = f"{form_id}_p{page_num}"
        
        # Store metadata for later
        forms_metadata[form_id] = (output_path, page_num, pdf_path)
        
        # Request 1: Main text extraction
        batch_requests.append(create_batch_request(
            custom_id=f"{form_id}_main",
            image_base64=image_base64,
            system_prompt=SYSTEM_PROMPT_MAIN,
            user_prompt=USER_PROMPT_MAIN
        ))
        
        # Request 2: Contract type checkboxes
        batch_requests.append(create_batch_request(
            custom_id=f"{form_id}_contract_type",
            image_base64=image_base64,
            system_prompt=SYSTEM_PROMPT_CONTRACT_TYPE,
            user_prompt=USER_PROMPT_CONTRACT_TYPE
        ))
        
        # Request 3: Yes/No fields and vendor status
        batch_requests.append(create_batch_request(
            custom_id=f"{form_id}_yesno",
            image_base64=image_base64,
            system_prompt=SYSTEM_PROMPT_YES_NO,
            user_prompt=USER_PROMPT_YES_NO_FIELDS
        ))
    
    if skipped_count > 0:
        print(f"\nWarning: Skipped {skipped_count} corrupted or unreadable PDF(s)")
    
    return batch_requests, forms_metadata


def estimate_request_size(request: Dict) -> int:
    """Estimate the size of a request in bytes."""
    return len(json.dumps(request).encode('utf-8'))


def chunk_requests(requests: List[Dict], max_size_mb: int = 200) -> List[List[Dict]]:
    """
    Split requests into chunks that are under the size limit.
    Uses 200MB as default to leave margin below the 256MB API limit.
    
    Args:
        requests: List of all requests
        max_size_mb: Maximum size per chunk in MB
    
    Returns:
        List of request chunks
    """
    max_size_bytes = max_size_mb * 1024 * 1024
    chunks = []
    current_chunk = []
    current_size = 0
    
    for request in requests:
        request_size = estimate_request_size(request)
        
        # If adding this request would exceed the limit, start a new chunk
        if current_size + request_size > max_size_bytes and current_chunk:
            chunks.append(current_chunk)
            current_chunk = []
            current_size = 0
        
        current_chunk.append(request)
        current_size += request_size
    
    # Add the last chunk if it has any requests
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks


def submit_batch(client: anthropic.Anthropic, requests: List[Dict], 
                forms_metadata: Dict, batch_name: str = "") -> str:
    """
    Submit batch requests to Claude API and register in central registry.
    
    Args:
        client: Anthropic client
        requests: List of batch request dicts
        forms_metadata: Dict mapping form_id -> (output_path, page_num, pdf_path)
        batch_name: Optional suffix for batch file name
    
    Returns:
        batch_id for tracking
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Write requests to JSONL file for record-keeping
    batch_file = Path(CONFIG['output_dir']) / f"batch_{timestamp}{batch_name}.jsonl"
    
    with open(batch_file, 'w') as f:
        for req in requests:
            f.write(json.dumps(req) + '\n')
    
    print(f"Created batch file: {batch_file.name}")
    print(f"Total requests: {len(requests)}")
    
    # Estimate size
    total_size = sum(estimate_request_size(req) for req in requests)
    print(f"Estimated size: {total_size / (1024*1024):.2f} MB")
    
    # Submit to API
    batch = client.messages.batches.create(requests=requests)
    
    print(f"Batch submitted! ID: {batch.id}")
    print(f"Status: {batch.processing_status}\n")
    
    # Save batch metadata file (legacy compatibility)
    batch_metadata_file = batch_file.with_suffix('.meta.json')
    metadata = {
        'batch_id': batch.id,
        'batch_file': batch_file.name,
        'timestamp': timestamp,
        'num_requests': len(requests),
        'num_forms': len(forms_metadata),
        'status': batch.processing_status,
        'created_at': batch.created_at.isoformat() if batch.created_at else None
    }
    with open(batch_metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved batch metadata to: {batch_metadata_file.name}")
    
    # Register in central registry
    register_batch(
        batch_id=batch.id,
        batch_file=batch_file.name,
        timestamp=timestamp,
        status=batch.processing_status,
        created_at=batch.created_at.isoformat() if batch.created_at else None,
        forms_metadata=forms_metadata,
        num_requests=len(requests)
    )
    
    return batch.id


def submit_batches_chunked(client: anthropic.Anthropic, all_requests: List[Dict],
                          forms_metadata: Dict) -> List[str]:
    """
    Submit requests in multiple batches if needed to stay under size limits.
    
    Args:
        client: Anthropic client
        all_requests: List of all request dicts
        forms_metadata: Dict mapping form_id -> (output_path, page_num, pdf_path)
    
    Returns:
        List of batch_ids
    """
    # Split into chunks
    chunks = chunk_requests(all_requests, max_size_mb=200)
    
    print(f"\n{'='*60}")
    print(f"Splitting {len(all_requests)} requests into {len(chunks)} batch(es)")
    print(f"{'='*60}\n")
    
    batch_ids = []
    
    for i, chunk in enumerate(chunks, 1):
        print(f"Submitting batch {i}/{len(chunks)}...")
        batch_name = f"_part{i}" if len(chunks) > 1 else ""
        
        # Create metadata subset for this chunk
        chunk_form_ids = set()
        for req in chunk:
            custom_id = req['custom_id']
            if '_main' in custom_id:
                form_id = custom_id.replace('_main', '')
                chunk_form_ids.add(form_id)
        
        chunk_metadata = {fid: forms_metadata[fid] for fid in chunk_form_ids if fid in forms_metadata}
        
        batch_id = submit_batch(client, chunk, chunk_metadata, batch_name)
        batch_ids.append(batch_id)
    
    return batch_ids


def check_batch_status(client: anthropic.Anthropic, batch_id: str) -> Dict:
    """Check status of a batch."""
    batch = client.messages.batches.retrieve(batch_id)
    return {
        'status': batch.processing_status,
        'created_at': batch.created_at,
        'request_counts': {
            'processing': batch.request_counts.processing,
            'succeeded': batch.request_counts.succeeded,
            'errored': batch.request_counts.errored,
            'canceled': batch.request_counts.canceled,
            'expired': batch.request_counts.expired
        }
    }


def retrieve_batch_results(client: anthropic.Anthropic, batch_id: str) -> List[Dict]:
    """Retrieve results from a completed batch."""
    results = []
    
    for result in client.messages.batches.results(batch_id):
        results.append(result)
    
    return results


print("Batch processing functions loaded with registry tracking.")

Batch processing functions loaded with registry tracking.


## 5. Result Merging and Output

In [93]:
def clean_json_response(response_text: str) -> str:
    """
    Clean response text to extract just the JSON.
    Removes markdown code blocks, explanations, and extra text.
    """
    # Remove markdown code blocks
    if '```json' in response_text:
        # Extract content between ```json and ```
        start = response_text.find('```json') + 7
        end = response_text.find('```', start)
        response_text = response_text[start:end].strip()
    elif '```' in response_text:
        # Extract content between ``` and ```
        start = response_text.find('```') + 3
        end = response_text.find('```', start)
        response_text = response_text[start:end].strip()
    
    # Find the JSON object boundaries
    # Look for the first { and last }
    first_brace = response_text.find('{')
    last_brace = response_text.rfind('}')
    
    if first_brace != -1 and last_brace != -1:
        response_text = response_text[first_brace:last_brace+1]
    
    return response_text.strip()


def has_api_credit_error_in_result(result_dict: Dict) -> bool:
    """
    Check if a result dictionary contains an API credit error.
    """
    if 'error' in result_dict:
        error_msg = str(result_dict['error'])
        if 'credit balance is too low' in error_msg:
            return True
    return False


def merge_results(main_result: Dict, contract_type_result: Dict, yesno_result: Dict, 
                  pdf_path: str, page_num: int) -> Dict:
    """
    Merge results from all three queries into final output.
    """
    merged = {
        'metadata': {
            'source_file': str(pdf_path),
            'page_number': page_num,
            'extracted_at': datetime.now().isoformat(),
            'model': CONFIG['model']
        },
        'extracted_data': {}
    }
    
    # Add main extraction results
    if main_result:
        merged['extracted_data'].update(main_result)
    
    # Add contract type
    if contract_type_result:
        merged['extracted_data']['contract_type_info'] = contract_type_result
    
    # Add yes/no fields and vendor status
    if yesno_result:
        merged['extracted_data']['checkbox_fields'] = yesno_result
    
    return merged


def process_batch_results(results: List[Dict], forms_metadata: Dict[str, Tuple[str, int]]):
    """
    Process batch results and save individual JSON files.
    Skips forms with API credit errors to avoid infinite retrieval loops.
    
    Args:
        results: List of batch result objects
        forms_metadata: Dict mapping form_id -> (output_path, page_num)
    """
    # Group results by form
    results_by_form = {}
    
    for result in results:
        custom_id = result.custom_id
        
        # Parse form_id and query type
        if '_main' in custom_id:
            form_id = custom_id.replace('_main', '')
            query_type = 'main'
        elif '_contract_type' in custom_id:
            form_id = custom_id.replace('_contract_type', '')
            query_type = 'contract_type'
        elif '_yesno' in custom_id:
            form_id = custom_id.replace('_yesno', '')
            query_type = 'yesno'
        else:
            continue
        
        if form_id not in results_by_form:
            results_by_form[form_id] = {}
        
        # Extract the response content
        if result.result.type == 'succeeded':
            content = result.result.message.content[0].text
            
            # Clean the response to remove markdown and explanations
            cleaned_content = clean_json_response(content)
            
            try:
                # Parse JSON response
                parsed = json.loads(cleaned_content)
                results_by_form[form_id][query_type] = parsed
            except json.JSONDecodeError as e:
                print(f"Warning: Could not parse JSON for {custom_id}")
                print(f"  Error: {e}")
                print(f"  Cleaned content preview: {cleaned_content[:200]}...")
                results_by_form[form_id][query_type] = {'error': 'JSON parse failed', 'raw_response': cleaned_content}
        else:
            print(f"Error in {custom_id}: {result.result.error}")
            results_by_form[form_id][query_type] = {'error': str(result.result.error)}
    
    # Save merged results
    skipped_api_errors = 0
    saved_count = 0
    
    for form_id, form_results in results_by_form.items():
        if form_id not in forms_metadata:
            print(f"Warning: No metadata for {form_id}")
            continue
        
        # Check if ANY of the results contain an API credit error
        has_api_error = False
        for query_type in ['main', 'contract_type', 'yesno']:
            result = form_results.get(query_type, {})
            if has_api_credit_error_in_result(result):
                has_api_error = True
                break
        
        # Skip saving if this form has API credit errors
        if has_api_error:
            skipped_api_errors += 1
            if skipped_api_errors <= 10:
                print(f"Skipped (API credit error): {form_id}")
            continue
        
        output_path, page_num, pdf_path = forms_metadata[form_id]
        
        merged = merge_results(
            main_result=form_results.get('main', {}),
            contract_type_result=form_results.get('contract_type', {}),
            yesno_result=form_results.get('yesno', {}),
            pdf_path=pdf_path,
            page_num=page_num
        )
        
        save_result(output_path, merged)
        saved_count += 1
        if saved_count <= 10 or saved_count % 100 == 0:
            print(f"Saved: {output_path}")
    
    # Print summary
    if skipped_api_errors > 0:
        print(f"\n{'='*60}")
        print(f"Skipped {skipped_api_errors} form(s) with API credit errors")
        print(f"These will NOT be saved and can be re-extracted later")
        print(f"{'='*60}")


print("Result processing functions loaded.")

Result processing functions loaded.


## 6. Main Processing Pipeline

In [95]:
def load_prof_services_filenames() -> Set[str]:
    """
    Load the set of professional services contract filenames from CSV.
    Extracts base filenames from the pdfUrl column.
    Returns: Set of base filenames (without extension)
    """
    prof_csv_path = Path(CONFIG['prof_contract_csv'])
    
    if not prof_csv_path.exists():
        print(f"Warning: Professional services CSV not found: {prof_csv_path}")
        return set()
    
    df = pd.read_csv(prof_csv_path)
    
    # Extract filenames from pdfUrl column
    filenames = set()
    for url in df['pdfUrl']:
        # Extract filename from URL (e.g., "26143-000.pdf" from full URL)
        filename = url.split('/')[-1]
        # Remove .pdf extension to get base name
        base_name = filename.replace('.pdf', '')
        filenames.add(base_name)
    
    print(f"Loaded {len(filenames)} professional services contract filenames")
    return filenames


def get_forms_to_process() -> List[Tuple[str, str, int]]:
    """
    Get list of forms to process based on mode.
    Skips PDFs that:
    1. Already have output files (unless clobber=True)
    2. Are in pending batches (waiting for retrieval)

    Returns: List of (pdf_path, output_path, page_num) tuples
    """
    forms = []
    skipped_pending = 0
    skipped_processed = 0

    # Load professional services filenames if filtering is enabled
    prof_services_files = set()
    if CONFIG['prof_services_only']:
        prof_services_files = load_prof_services_filenames()
        if not prof_services_files:
            print("Warning: prof_services_only is True but no files loaded from CSV!")

    if CONFIG['mode'] == 'test':
        # Test mode: single-page PDFs
        test_dir = Path(CONFIG['test_dir'])
        if not test_dir.exists():
            print(f"Warning: Test directory not found: {test_dir}")
            return []

        for pdf_file in test_dir.glob('*.pdf'):
            # Check if filtering by professional services
            if CONFIG['prof_services_only']:
                base_name = pdf_file.stem
                if base_name not in prof_services_files:
                    continue

            pdf_path = str(pdf_file)
            page_num = 1
            # For page 1, no suffix (for backwards compatibility)
            output_name = f"{pdf_file.stem}.json"
            output_path = str(Path(CONFIG['output_dir']) / output_name)

            # Check if already in a pending batch
            if is_pdf_in_pending_batch(pdf_path, page_num):
                skipped_pending += 1
                continue

            # Check if already processed
            if should_process(output_path, CONFIG['clobber']):
                forms.append((pdf_path, output_path, page_num))

                # Apply limit if set
                if CONFIG['limit_files'] and len(forms) >= CONFIG['limit_files']:
                    print(f"Reached file limit ({CONFIG['limit_files']}), stopping scan")
                    break
            else:
                skipped_processed += 1

    elif CONFIG['mode'] == 'production':
        # Production mode: multi-page PDFs with metadata CSV
        metadata_path = Path(CONFIG['metadata_csv'])
        if not metadata_path.exists():
            print(f"Error: Metadata CSV not found: {metadata_path}")
            return []

        # Add low_memory=False to suppress dtype warning
        df = pd.read_csv(metadata_path, low_memory=False)
        prod_dir = Path(CONFIG['production_dir'])

        for _, row in df.iterrows():
            # Get form_pages value
            form_pages = str(row['form_pages']).strip()

            # Skip if form_pages is empty
            if not form_pages or form_pages == '' or form_pages == 'nan':
                continue

            # Parse comma-separated page numbers and get the first (lowest) one
            try:
                page_numbers = [int(p.strip()) for p in form_pages.split(',') if p.strip()]
                if not page_numbers:
                    continue
                page_num = min(page_numbers)  # Get the first/lowest page number
            except ValueError:
                print(f"Warning: Could not parse form_pages '{form_pages}' for {row['filename']}")
                continue

            pdf_path = prod_dir / row['filename']

            # Check if filtering by professional services
            if CONFIG['prof_services_only']:
                base_name = Path(row['filename']).stem
                if base_name not in prof_services_files:
                    continue  # Skip silently to avoid spam

            if not pdf_path.exists():
                print(f"Warning: File not found: {pdf_path}")
                continue

            # Generate output name - NO suffix for page 1 (backwards compatibility)
            if page_num == 1:
                output_name = f"{pdf_path.stem}.json"
            else:
                output_name = f"{pdf_path.stem}_p{page_num}.json"
            output_path = str(Path(CONFIG['output_dir']) / output_name)

            # Check if already in a pending batch
            if is_pdf_in_pending_batch(str(pdf_path), page_num):
                skipped_pending += 1
                continue

            # Check if already processed
            if should_process(output_path, CONFIG['clobber']):
                forms.append((str(pdf_path), output_path, page_num))

                # Apply limit if set
                if CONFIG['limit_files'] and len(forms) >= CONFIG['limit_files']:
                    print(f"Reached file limit ({CONFIG['limit_files']}), stopping scan")
                    break
            else:
                skipped_processed += 1

    # Print skip summary
    if skipped_pending > 0:
        print(f"Skipped {skipped_pending} PDF(s) already in pending batches")
    if skipped_processed > 0:
        print(f"Skipped {skipped_processed} PDF(s) already processed")

    return forms


def run_extraction_pipeline():
    """
    Main pipeline: prepare requests, submit batch(es), wait for results, save output.
    """
    print("=" * 60)
    print("EDS FORM EXTRACTION PIPELINE")
    print("=" * 60)
    
    # Initialize client
    if not CONFIG['api_key']:
        print("ERROR: ANTHROPIC_API_KEY not set!")
        print("Please set it: os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'")
        return
    
    client = anthropic.Anthropic(api_key=CONFIG['api_key'])
    
    # Get forms to process
    print(f"\nScanning for forms (mode: {CONFIG['mode']})...")
    if CONFIG['prof_services_only']:
        print("Filtering for PROFESSIONAL SERVICES CONTRACTS ONLY")
    if CONFIG['limit_files']:
        print(f"Limiting to {CONFIG['limit_files']} files")
    
    forms_to_process = get_forms_to_process()
    
    if not forms_to_process:
        print("No forms to process!")
        return
    
    print(f"Found {len(forms_to_process)} forms to process")
    print(f"This will create {len(forms_to_process) * 3} API requests\n")
    
    # Prepare batch requests
    print("Preparing batch requests...")
    batch_requests, forms_metadata = prepare_batch_requests(forms_to_process)
    
    # Submit batch(es) with chunking
    print("\nSubmitting to Claude Batch API...")
    batch_ids = submit_batches_chunked(client, batch_requests, forms_metadata)
    
    # Monitor status for all batches
    print("\nMonitoring batch status (this may take up to 24 hours)...")
    print("You can close this notebook and check back later.")
    print(f"Batch IDs: {batch_ids}\n")
    
    all_completed = False
    check_interval = CONFIG['batch_check_interval_seconds']
    
    while not all_completed:
        all_completed = True
        
        for i, batch_id in enumerate(batch_ids, 1):
            status = check_batch_status(client, batch_id)
            print(f"Batch {i}/{len(batch_ids)} ({batch_id}):")
            print(f"  Status: {status['status']}")
            print(f"  Succeeded: {status['request_counts']['succeeded']}")
            print(f"  Processing: {status['request_counts']['processing']}")
            print(f"  Errored: {status['request_counts']['errored']}")
            
            if status['status'] != 'ended':
                all_completed = False
        
        if not all_completed:
            interval_minutes = check_interval // 60
            interval_seconds = check_interval % 60
            if interval_seconds == 0:
                time_str = f"{interval_minutes} minute{'s' if interval_minutes != 1 else ''}"
            else:
                time_str = f"{check_interval} seconds"
            print(f"\nStill processing... checking again in {time_str}")
            time.sleep(check_interval)
        else:
            print("\nAll batches complete!")
    
    # Retrieve and process results from all batches
    print("\nRetrieving results from all batches...")
    all_results = []
    
    for i, batch_id in enumerate(batch_ids, 1):
        print(f"Retrieving batch {i}/{len(batch_ids)}...")
        results = retrieve_batch_results(client, batch_id)
        all_results.extend(results)
    
    print(f"\nProcessing {len(all_results)} total results...")
    process_batch_results(all_results, forms_metadata)
    
    print("\n" + "=" * 60)
    print("EXTRACTION COMPLETE!")
    print(f"Results saved to: {CONFIG['output_dir']}")
    print("=" * 60)


print("Pipeline functions loaded. Ready to run!")

Pipeline functions loaded. Ready to run!


## 7. Run the Pipeline

Execute this cell to start processing!

In [97]:
# Make sure to set your API key first!
#os.environ['ANTHROPIC_API_KEY'] = api_key
run_extraction_pipeline()

EDS FORM EXTRACTION PIPELINE

Scanning for forms (mode: production)...
Filtering for PROFESSIONAL SERVICES CONTRACTS ONLY
Loaded 35300 professional services contract filenames
Skipped 25049 PDF(s) already processed
Found 1059 forms to process
This will create 3177 API requests

Preparing batch requests...
Preparing: 47737-000.pdf (page 1)
  Image: 1222x1568 (~2554 tokens)
Preparing: 47872-000.pdf (page 1)
  Image: 1220x1568 (~2550 tokens)
Preparing: 48521-001.pdf (page 1)
  Image: 1219x1568 (~2548 tokens)
Preparing: 47879-000.pdf (page 1)
  Image: 1214x1568 (~2538 tokens)
Preparing: 46538-000.pdf (page 1)
  Image: 1193x1568 (~2494 tokens)
Preparing: 47494-000.pdf (page 1)
  Image: 1204x1568 (~2517 tokens)
Preparing: 48511-001.pdf (page 1)
  Image: 1214x1568 (~2538 tokens)
Preparing: 47903-000.pdf (page 1)
  Image: 1210x1568 (~2529 tokens)
Preparing: 47881-000.pdf (page 1)
  Image: 1220x1568 (~2550 tokens)
Preparing: 48262-001.pdf (page 1)
  Image: 1223x1568 (~2556 tokens)
Preparing: 47

## 8. Utility: Check Batch Status Later

If you need to check on a batch later, use this cell.

In [99]:
# List all batches from Anthropic API and match them to your local batch files

client = anthropic.Anthropic(api_key=CONFIG['api_key'])

print("Fetching all batches from Anthropic API...")
print("=" * 60)

# Get all batches (API returns most recent first, max 100)
try:
    batches_response = client.messages.batches.list(limit=100)
    batches = batches_response.data
    
    print(f"\nFound {len(batches)} batch(es) from API\n")
    
    # Load your local batch files
    output_dir = Path(CONFIG['output_dir'])
    local_batch_files = sorted(output_dir.glob('batch_*.jsonl'))
    
    print(f"You have {len(local_batch_files)} local batch file(s)")
    print("=" * 60)
    
    # Show all batches with their details
    for i, batch in enumerate(batches, 1):
        # Convert datetime to string
        created_at_str = batch.created_at.isoformat() if batch.created_at else "unknown"
        created_date = created_at_str[:10] if len(created_at_str) >= 10 else "unknown"
        created_time = created_at_str[11:19] if len(created_at_str) > 11 else ""
        
        total_requests = (batch.request_counts.succeeded + 
                         batch.request_counts.processing + 
                         batch.request_counts.errored)
        
        print(f"\n{i}. Batch ID: {batch.id}")
        print(f"   Created: {created_date} at {created_time}")
        print(f"   Status: {batch.processing_status}")
        print(f"   Requests: {total_requests}")
        print(f"   Succeeded: {batch.request_counts.succeeded}")
        print(f"   Errored: {batch.request_counts.errored}")
        
        # Try to match with local files by timestamp
        if batch.created_at:
            # Extract date and time from datetime object
            # Convert to our format: 20251205_010818
            date_part = batch.created_at.strftime('%Y%m%d')  # 20251205
            time_part = batch.created_at.strftime('%H%M%S')  # 010818
            timestamp = f"{date_part}_{time_part}"
            
            # Find matching files - the timestamp appears after "batch_" prefix
            # Files look like: batch_20251205_010818_part1.jsonl
            matching_files = [f for f in local_batch_files if f"batch_{timestamp}" in f.name]
            
            if matching_files:
                print(f"   → MATCHES: {', '.join([f.name for f in matching_files])}")
            else:
                # Try with just the date in case timing is slightly off
                matching_by_date = [f for f in local_batch_files if date_part in f.name]
                if matching_by_date:
                    print(f"   → Possible matches by date: {', '.join([f.name for f in matching_by_date[:3]])}")
                else:
                    print(f"   → No matching local file found (looking for batch_{timestamp})")
    
    print("\n" + "=" * 60)
    print("To create metadata files for batches without IDs:")
    print("Copy the batch IDs above and create .meta.json files")
    print("Or copy-paste the batch IDs to match with your local files")
    print("=" * 60)
    
except Exception as e:
    import traceback
    print(f"Error fetching batches: {e}")
    print(f"Traceback: {traceback.format_exc()}")
    print("\nNote: The API may have limits on how far back it shows batches.")

Fetching all batches from Anthropic API...

Found 100 batch(es) from API

You have 180 local batch file(s)

1. Batch ID: msgbatch_013b4pg2Uh6KcJ8tSUtFTPXK
   Created: 2025-12-30 at 20:33:13
   Status: ended
   Requests: 3
   Succeeded: 3
   Errored: 0
   → Possible matches by date: batch_20251230_123114_part1.jsonl, batch_20251230_123125_part2.jsonl, batch_20251230_123142_part3.jsonl

2. Batch ID: msgbatch_01WsV7cenAFe98DALdSDV2LD
   Created: 2025-12-30 at 20:33:10
   Status: ended
   Requests: 449
   Succeeded: 449
   Errored: 0
   → Possible matches by date: batch_20251230_123114_part1.jsonl, batch_20251230_123125_part2.jsonl, batch_20251230_123142_part3.jsonl

3. Batch ID: msgbatch_017ruf1GoQZkzGcVx94wZ2tz
   Created: 2025-12-30 at 20:32:50
   Status: ended
   Requests: 446
   Succeeded: 446
   Errored: 0
   → Possible matches by date: batch_20251230_123114_part1.jsonl, batch_20251230_123125_part2.jsonl, batch_20251230_123142_part3.jsonl

4. Batch ID: msgbatch_016fTwZABQrdaqL7xgBA6J

In [101]:
# AUTO-MATCH: Match remaining batch IDs to batch files by creation order (12/5 only)

import traceback

print("AUTO-MATCHING BATCH IDs TO FILES BY CREATION ORDER")
print("=" * 60)
print("Filtering: Only batches from 12/5/2025")
print("Matching: Based on creation order (earliest ↔ earliest)")
print("=" * 60)

try:
    client = anthropic.Anthropic(api_key=CONFIG['api_key'])
    
    # Get all batches from API
    print("\nFetching batches from API...")
    batches_response = client.messages.batches.list(limit=100)
    all_batches = batches_response.data
    
    # Filter to only 12/5/2025 batches, sorted by creation time (oldest first)
    batches_12_5 = []
    for batch in all_batches:
        if batch.created_at:
            created_date = batch.created_at.strftime('%Y-%m-%d')
            if created_date >= '2025-12-05':
                batches_12_5.append(batch)
    
    # Sort by creation time (oldest first)
    batches_12_5.sort(key=lambda b: b.created_at)
    
    print(f"Found {len(batches_12_5)} batches from 12/5/2025 onwards")
    
    # Get local batch files without metadata, sorted by file creation time (oldest first)
    output_dir = Path(CONFIG['output_dir'])
    
    # Load existing batch IDs to skip already-matched files
    batch_ids_map = {}
    for meta_file in output_dir.glob('batch_*.meta.json'):
        try:
            with open(meta_file, 'r') as f:
                metadata = json.load(f)
                batch_filename = metadata.get('batch_file')
                if batch_filename:
                    batch_ids_map[batch_filename] = True
        except:
            pass
    
    # Find batch files without metadata
    unmatched_files = []
    for batch_file in output_dir.glob('batch_20251205_*.jsonl'):
        if batch_file.name not in batch_ids_map:
            # Get file creation time
            file_stat = batch_file.stat()
            creation_time = file_stat.st_birthtime  # macOS creation time
            unmatched_files.append((batch_file, creation_time))
    
    # Sort by creation time (oldest first)
    unmatched_files.sort(key=lambda x: x[1])
    
    print(f"Found {len(unmatched_files)} local batch files without metadata")
    
    if len(batches_12_5) != len(unmatched_files):
        print(f"\n⚠ WARNING: Batch count mismatch!")
        print(f"  API batches (12/5): {len(batches_12_5)}")
        print(f"  Local files: {len(unmatched_files)}")
        print(f"\nProceeding anyway - will match as many as possible")
    
    # Match batches to files by order
    matches = []
    num_to_match = min(len(batches_12_5), len(unmatched_files))
    
    print(f"\n{'='*60}")
    print(f"MATCHING {num_to_match} BATCH ID(s) TO FILE(s)")
    print(f"{'='*60}\n")
    
    for i in range(num_to_match):
        batch = batches_12_5[i]
        batch_file, _ = unmatched_files[i]
        
        # Display the match
        created_time = batch.created_at.strftime('%Y-%m-%d %H:%M:%S')
        print(f"{i+1}. {batch_file.name}")
        print(f"   ↔ {batch.id}")
        print(f"   Created: {created_time}")
        print(f"   Status: {batch.processing_status}")
        
        matches.append((batch_file, batch.id, batch))
    
    # Ask user to confirm
    print(f"\n{'='*60}")
    print(f"READY TO CREATE {len(matches)} METADATA FILE(s)")
    print(f"{'='*60}")
    print("\nThis will create .meta.json files for the matched batches.")
    print("Then you can use cell-19 to retrieve results automatically.")
    
    # Auto-proceed (you can change this to manual confirmation if preferred)
    proceed = True  # Set to False if you want to manually confirm first
    
    if proceed:
        print("\nCreating metadata files...")
        
        for batch_file, batch_id, batch in matches:
            # Read the batch file to get request count
            num_requests = 0
            with open(batch_file, 'r') as f:
                for line in f:
                    if line.strip():
                        num_requests += 1
            
            # Create metadata
            timestamp = batch_file.stem.replace('batch_', '').split('_part')[0]
            
            metadata = {
                'batch_id': batch_id,
                'batch_file': batch_file.name,
                'timestamp': timestamp,
                'num_requests': num_requests,
                'num_forms': num_requests // 3,
                'status': batch.processing_status,
                'created_at': batch.created_at.isoformat() if batch.created_at else None
            }
            
            # Save metadata file
            meta_file = batch_file.with_suffix('.meta.json')
            with open(meta_file, 'w') as f:
                json.dump(metadata, f, indent=2)
            
            print(f"  ✓ Created: {meta_file.name}")
        
        print(f"\n{'='*60}")
        print(f"✓ COMPLETE! Created {len(matches)} metadata file(s)")
        print(f"{'='*60}")
        print("\nNow run cell-19 to retrieve results automatically!")
    else:
        print("\nSet proceed = True in the cell to create metadata files")
    
except Exception as e:
    print(f"\n✗ Error: {e}")
    print(f"\nFull traceback:")
    print(traceback.format_exc())

AUTO-MATCHING BATCH IDs TO FILES BY CREATION ORDER
Filtering: Only batches from 12/5/2025
Matching: Based on creation order (earliest ↔ earliest)

Fetching batches from API...
Found 100 batches from 12/5/2025 onwards
Found 0 local batch files without metadata

⚠ WARNING: Batch count mismatch!
  API batches (12/5): 100
  Local files: 0

Proceeding anyway - will match as many as possible

MATCHING 0 BATCH ID(s) TO FILE(s)


READY TO CREATE 0 METADATA FILE(s)

This will create .meta.json files for the matched batches.
Then you can use cell-19 to retrieve results automatically.

Creating metadata files...

✓ COMPLETE! Created 0 metadata file(s)

Now run cell-19 to retrieve results automatically!


## 9. Utility: Retrieve Completed Batch

If a batch is complete and you need to retrieve results again.

In [103]:
# Retrieve and process results from all completed batches - AUTOMATIC

import re

def load_batch_ids_from_metadata() -> Dict[str, str]:
    """
    Load batch IDs from .meta.json files automatically.
    Returns: Dict mapping batch_filename -> batch_id
    """
    output_dir = Path(CONFIG['output_dir'])
    batch_ids = {}
    
    for meta_file in output_dir.glob('batch_*.meta.json'):
        try:
            with open(meta_file, 'r') as f:
                metadata = json.load(f)
                batch_filename = metadata.get('batch_file')
                batch_id = metadata.get('batch_id')
                if batch_filename and batch_id:
                    batch_ids[batch_filename] = batch_id
        except Exception as e:
            print(f"Warning: Could not read {meta_file.name}: {e}")
    
    return batch_ids


def find_unprocessed_batches() -> List[Tuple[Path, int, int, Optional[str]]]:
    """
    Find batch files that haven't been retrieved yet.
    Returns: List of (batch_file, total_forms, processed_forms, batch_id) tuples
    """
    output_dir = Path(CONFIG['output_dir'])
    
    # Load all batch IDs from metadata files
    batch_ids_map = load_batch_ids_from_metadata()
    
    unprocessed = []
    
    # Get pending batch IDs from registry
    pending_batch_ids = get_pending_batches()
    
    for batch_file in sorted(output_dir.glob('batch_*.jsonl')):
        try:
            # Get batch ID from metadata if available
            batch_id = batch_ids_map.get(batch_file.name)
            
            # Skip if this batch has already been retrieved
            if batch_id and batch_id not in pending_batch_ids:
                continue
            
            form_ids_in_batch = []
            with open(batch_file, 'r') as f:
                for line in f:
                    if not line.strip():
                        continue
                    request = json.loads(line)
                    custom_id = request.get('custom_id', '')
                    # Extract form_id from custom_id (only count _main to avoid triplicates)
                    if '_main' in custom_id:
                        form_id = custom_id.replace('_main', '')
                        form_ids_in_batch.append(form_id)
            
            # Count how many have output files
            processed_count = 0
            for form_id in form_ids_in_batch:
                output_file = output_dir / f"{form_id}.json"
                if output_file.exists():
                    processed_count += 1
            
            total_count = len(form_ids_in_batch)
            
            # If batch hasn't been retrieved yet, add to unprocessed list
            if batch_id and batch_id in pending_batch_ids:
                unprocessed.append((batch_file, total_count, processed_count, batch_id))
                
        except Exception as e:
            print(f"Warning: Error checking {batch_file.name}: {e}")
    
    return unprocessed


def recreate_forms_metadata_from_batch(batch_file: Path) -> Dict[str, Tuple[str, int, str]]:
    """
    Recreate forms_metadata dict from a batch JSONL file.
    Returns: Dict mapping form_id -> (output_path, page_num, pdf_path)
    """
    forms_metadata = {}
    
    try:
        with open(batch_file, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                
                request = json.loads(line)
                custom_id = request.get('custom_id', '')
                
                # Only process _main requests to avoid duplicates
                if not custom_id.endswith('_main'):
                    continue
                
                # Extract form_id (remove _main suffix)
                form_id = custom_id.replace('_main', '')
                
                # Parse form_id to get filename and page number
                if '_p' in form_id:
                    # Format: "filename_p2" -> filename="filename", page_num=2
                    base_name, page_part = form_id.rsplit('_p', 1)
                    page_num = int(page_part)
                else:
                    # Format: "filename" -> page_num=1
                    base_name = form_id
                    page_num = 1
                
                # Reconstruct paths
                pdf_filename = f"{base_name}.pdf"
                if CONFIG['mode'] == 'production':
                    pdf_path = str(Path(CONFIG['production_dir']) / pdf_filename)
                else:
                    pdf_path = str(Path(CONFIG['test_dir']) / pdf_filename)
                
                output_path = str(Path(CONFIG['output_dir']) / f"{form_id}.json")
                
                forms_metadata[form_id] = (output_path, page_num, pdf_path)
    
    except Exception as e:
        print(f"Error recreating metadata from {batch_file.name}: {e}")
    
    return forms_metadata


# Main execution
print("Checking for unprocessed batches...")
print("=" * 60)

# Find unprocessed batches (with batch IDs automatically loaded)
unprocessed_batches = find_unprocessed_batches()

if not unprocessed_batches:
    print("\n✓ All batches have been retrieved!")
    print("No batch results to retrieve.")
else:
    # Separate batches with and without batch IDs
    batches_with_ids = [(b, t, p, bid) for b, t, p, bid in unprocessed_batches if bid]
    batches_without_ids = [(b, t, p) for b, t, p, bid in unprocessed_batches if not bid]
    
    if batches_without_ids:
        print(f"\n⚠ Found {len(batches_without_ids)} batch(es) WITHOUT batch IDs:")
        print("These batches were created before metadata tracking was added.\n")
        for batch_file, total, processed in batches_without_ids:
            print(f"  - {batch_file.name}")
            print(f"    Progress: {processed}/{total} forms processed")
        print("\nYou'll need to manually add batch IDs for these batches.")
        print("=" * 60)
    
    if batches_with_ids:
        print(f"\n✓ Found {len(batches_with_ids)} batch(es) ready to retrieve:\n")
        
        for i, (batch_file, total, processed, batch_id) in enumerate(batches_with_ids, 1):
            timestamp_match = re.search(r'batch_(\d{8}_\d{6})', batch_file.name)
            timestamp_str = ""
            if timestamp_match:
                timestamp = timestamp_match.group(1)
                timestamp_str = f" (created {timestamp[:8]} at {timestamp[9:11]}:{timestamp[11:13]}:{timestamp[13:15]})"
            
            print(f"  {i}. {batch_file.name}{timestamp_str}")
            print(f"     Progress: {processed}/{total} forms processed")
            print(f"     Batch ID: {batch_id}")
        
        print("\n" + "=" * 60)
        print("RETRIEVING RESULTS AUTOMATICALLY...")
        print("=" * 60)
        
        # Process all batches automatically
        client = anthropic.Anthropic(api_key=CONFIG['api_key'])
        
        for batch_file, total, processed, batch_id in batches_with_ids:
            print(f"\n{'='*60}")
            print(f"Processing: {batch_file.name}")
            print(f"Batch ID: {batch_id}")
            print(f"{'='*60}")
            
            try:
                # Check batch status first
                status = check_batch_status(client, batch_id)
                print(f"  Status: {status['status']}")
                print(f"  Succeeded: {status['request_counts']['succeeded']}")
                print(f"  Errored: {status['request_counts']['errored']}")
                
                if status['status'] != 'ended':
                    print(f"  ⚠ Batch not complete yet. Skipping...")
                    continue
                
                # Retrieve results
                print(f"  Retrieving results...")
                results = retrieve_batch_results(client, batch_id)
                print(f"  Retrieved {len(results)} results")
                
                # Recreate metadata from batch file
                forms_metadata = recreate_forms_metadata_from_batch(batch_file)
                print(f"  Recreated metadata for {len(forms_metadata)} forms")
                
                # Process results
                process_batch_results(results, forms_metadata)
                
                # IMPORTANT: Mark batch as retrieved to prevent re-processing
                mark_batch_retrieved(batch_id)
                
                print(f"  ✓ Results processed and batch marked as retrieved")
                
            except Exception as e:
                print(f"  ✗ Error: {e}")
        
        print("\n" + "=" * 60)
        print("BATCH RETRIEVAL COMPLETE")
        print("=" * 60)

Checking for unprocessed batches...

✓ Found 8 batch(es) ready to retrieve:

  1. batch_20251230_123114_part1.jsonl (created 20251230 at 12:31:14)
     Progress: 154/154 forms processed
     Batch ID: msgbatch_01EjfFVgQ1tzUdy5d7gyWGba
  2. batch_20251230_123125_part2.jsonl (created 20251230 at 12:31:25)
     Progress: 152/152 forms processed
     Batch ID: msgbatch_01DHkD9EUH4MYUdsgcddSapJ
  3. batch_20251230_123142_part3.jsonl (created 20251230 at 12:31:42)
     Progress: 152/152 forms processed
     Batch ID: msgbatch_01XCZWetqAB9L9J8nLAFdun6
  4. batch_20251230_123157_part4.jsonl (created 20251230 at 12:31:57)
     Progress: 152/152 forms processed
     Batch ID: msgbatch_01RsBH8jBYnknmGoMTYiLykx
  5. batch_20251230_123209_part5.jsonl (created 20251230 at 12:32:09)
     Progress: 150/150 forms processed
     Batch ID: msgbatch_016fTwZABQrdaqL7xgBA6JJa
  6. batch_20251230_123222_part6.jsonl (created 20251230 at 12:32:22)
     Progress: 149/149 forms processed
     Batch ID: msgbatch_

## 9b. One-Time Fix: Mark All Existing Batches as Retrieved

Run this cell ONCE to mark all existing batches in the registry as retrieved. This fixes the issue where old batches with API errors are stuck in "pending" state.

# ONE-TIME FIX: Mark all existing batches as retrieved

print("ONE-TIME FIX: Marking all existing batches as retrieved")
print("=" * 60)

registry = load_batch_registry()

if not registry['batches']:
    print("\nNo batches found in registry.")
else:
    print(f"\nFound {len(registry['batches'])} batch(es) in registry")
    
    marked_count = 0
    for batch_id, batch_info in registry['batches'].items():
        # Check if already marked
        if batch_info.get('retrieved', False):
            continue
        
        # Mark as retrieved
        batch_info['retrieved'] = True
        marked_count += 1
        
        if marked_count <= 10:
            print(f"  Marking batch {batch_id} as retrieved")
    
    # Save updated registry
    save_batch_registry(registry)
    
    print(f"\n{'='*60}")
    print(f"Marked {marked_count} batch(es) as retrieved")
    print(f"{'='*60}")
    print(f"\nThis fixes the issue where {len(get_pending_batches())} pending batch(es) ")
    print("were stuck in an infinite loop.")
    print("\nPDFs that had API credit errors can now be re-extracted in new batches.")

## 10. Cost Estimation

Estimate costs for your batch.

In [66]:
def estimate_batch_cost(num_forms: int, model: str = 'claude-haiku-4-5'):
    """
    Estimate cost for processing forms with optimized prompt caching.
    """
    # Pricing per million tokens
    pricing = {
        'claude-haiku-4-5': {'input': 1.00, 'output': 5.00},
        'claude-sonnet-4-5': {'input': 3.00, 'output': 15.00}
    }
    
    model_key = 'claude-haiku-4-5' if 'haiku' in model else 'claude-sonnet-4-5'
    
    # Token estimates per form (3 queries with optimized caching)
    image_tokens = 1280  # Typical form image (~1200x1568 @ 150dpi JPEG)
    
    # System prompts (NOW CACHED - includes all detailed instructions and schemas)
    system_prompt_main = 400       # Main extraction with field list
    system_prompt_contract = 300   # Contract type with options and rules
    system_prompt_yesno = 500      # Yes/No fields with detailed schema
    total_system_tokens = system_prompt_main + system_prompt_contract + system_prompt_yesno  # 1200 tokens
    
    # User prompts (NOT CACHED - now minimal)
    user_prompt_main = 12          # "Extract all text fields from this EDS form and return as JSON."
    user_prompt_contract = 10      # "Identify which contract type is selected on this form."
    user_prompt_yesno = 13         # "Identify the yes/no selections and vendor status checkboxes on this form."
    total_user_tokens = user_prompt_main + user_prompt_contract + user_prompt_yesno  # ~35 tokens
    
    output_tokens = 300  # Average response per query
    queries_per_form = 3
    
    # First form pays full price for system prompts (establishing cache)
    first_form_input = (image_tokens * queries_per_form) + total_system_tokens + total_user_tokens
    first_form_output = output_tokens * queries_per_form
    
    # Subsequent forms benefit from 90% cache savings on system prompts
    cached_system_tokens = total_system_tokens * 0.1  # 90% savings
    subsequent_form_input = (image_tokens * queries_per_form) + cached_system_tokens + total_user_tokens
    subsequent_form_output = output_tokens * queries_per_form
    
    # Total tokens
    if num_forms > 0:
        total_input = first_form_input + (subsequent_form_input * (num_forms - 1))
        total_output = first_form_output + (subsequent_form_output * (num_forms - 1))
    else:
        total_input = 0
        total_output = 0
    
    # Cost calculation (with 50% batch discount)
    input_cost = (total_input / 1_000_000) * pricing[model_key]['input'] * 0.5
    output_cost = (total_output / 1_000_000) * pricing[model_key]['output'] * 0.5
    total_cost = input_cost + output_cost
    
    print(f"Cost Estimate for {num_forms} forms using {model_key}:")
    print(f"  Token breakdown per form:")
    print(f"    - Images: {image_tokens * 3:,} tokens (3 queries)")
    print(f"    - System prompts: {total_system_tokens:,} tokens (CACHED after first form)")
    print(f"    - User prompts: {total_user_tokens:,} tokens (minimal, not cached)")
    print(f"\n  Total input tokens: {total_input:,}")
    print(f"  Total output tokens: {total_output:,}")
    print(f"  Input cost (with 50% batch discount): ${input_cost:.2f}")
    print(f"  Output cost (with 50% batch discount): ${output_cost:.2f}")
    print(f"  Total estimated cost: ${total_cost:.2f}")
    print(f"\n  Cost per form: ${total_cost/num_forms:.4f}")
    print(f"\n  Optimization benefits:")
    print(f"    - System prompts cached: {total_system_tokens:,} tokens × {num_forms-1:,} forms = {total_system_tokens * (num_forms-1):,} tokens saved")
    print(f"    - User prompts minimized: ~565 tokens saved per form")
    print(f"    - Combined with 50% batch discount and 90% prompt caching")


# Estimate for different scenarios
print("Small test (100 forms):")
estimate_batch_cost(100)

print("\n" + "="*60 + "\n")
print("Full dataset (40,000 forms):")
estimate_batch_cost(40000)

Small test (100 forms):
Cost Estimate for 100 forms using claude-haiku-4-5:
  Token breakdown per form:
    - Images: 3,840 tokens (3 queries)
    - System prompts: 1,200 tokens (CACHED after first form)
    - User prompts: 35 tokens (minimal, not cached)

  Total input tokens: 400,580.0
  Total output tokens: 90,000
  Input cost (with 50% batch discount): $0.20
  Output cost (with 50% batch discount): $0.22
  Total estimated cost: $0.43

  Cost per form: $0.0043

  Optimization benefits:
    - System prompts cached: 1,200 tokens × 99 forms = 118,800 tokens saved
    - User prompts minimized: ~565 tokens saved per form
    - Combined with 50% batch discount and 90% prompt caching


Full dataset (40,000 forms):
Cost Estimate for 40000 forms using claude-haiku-4-5:
  Token breakdown per form:
    - Images: 3,840 tokens (3 queries)
    - System prompts: 1,200 tokens (CACHED after first form)
    - User prompts: 35 tokens (minimal, not cached)

  Total input tokens: 159,801,080.0
  Total o

## 11. Cleanup: Remove API Error Files

After extraction runs, check for and remove files that failed due to API credit errors so they can be re-processed.

In [105]:
def has_api_credit_error(json_file: Path) -> bool:
    """
    Check if a JSON file contains an API credit error.
    """
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        # Check if there's an error in extracted_data
        if 'extracted_data' in data and 'error' in data['extracted_data']:
            error_msg = data['extracted_data']['error']
            # Check for API credit error message
            if 'credit balance is too low' in error_msg:
                return True
        
        return False
    except Exception as e:
        print(f"Error reading {json_file.name}: {e}")
        return False


# Find all JSON files in output directory
output_dir = Path(CONFIG['output_dir'])
all_json_files = list(output_dir.glob('*.json'))

# Filter out batch metadata files
all_json_files = [f for f in all_json_files if not f.name.startswith('batch_')]

print(f"Scanning {len(all_json_files)} JSON files for API credit errors...")
print("=" * 60)

# Identify files with API credit errors
api_error_files = []
for json_file in all_json_files:
    if has_api_credit_error(json_file):
        api_error_files.append(json_file)

print(f"\nFound {len(api_error_files)} files with API credit errors")

if api_error_files:
    print(f"\nFirst 20 files with API errors:")
    for f in api_error_files[:20]:
        print(f"  - {f.name}")
    
    print(f"\n{'='*60}")
    print(f"DELETING {len(api_error_files)} FILES")
    print(f"{'='*60}")
    
    deleted_count = 0
    failed_count = 0
    
    for json_file in api_error_files:
        try:
            json_file.unlink()  # Delete the file
            deleted_count += 1
            if deleted_count <= 10 or deleted_count % 100 == 0:
                print(f"Deleted: {json_file.name}")
        except Exception as e:
            print(f"Failed to delete {json_file.name}: {e}")
            failed_count += 1
    
    print(f"\n{'='*60}")
    print("CLEANUP SUMMARY")
    print(f"{'='*60}")
    print(f"Successfully deleted: {deleted_count}")
    print(f"Failed to delete: {failed_count}")
    print(f"\nThese {deleted_count} PDFs can now be re-extracted.")
    print(f"{'='*60}")
else:
    print("\n✓ No files with API credit errors found!")

Scanning 26108 JSON files for API credit errors...

Found 0 files with API credit errors

✓ No files with API credit errors found!
